# Family Genome × KEGG — Portfolio Walkthrough

다섯 가족 구성원(Father/Mother/Child 1/2/3)의 23andMe SNP 데이터를 NCBI ClinVar로 주석 달고,
KEGG의 질병-유전자-경로-약물 관계형 데이터와 교차검증한 프로젝트의 핵심 결과를
짧게 훑어본다. 전체 파이프라인을 다시 실행하지 않고, 이미 만들어진 `results/` 산출물을
그대로 불러와서 보여준다.

> 이 노트북은 가설 생성용 스크리닝 결과를 보여주는 것이지, 임상적 진단 도구가 아니다.
> 방법론은 [`docs/methodology.md`](../docs/methodology.md), 한계는
> [`docs/limitations.md`](../docs/limitations.md)를 참고할 것.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent
TABLES = ROOT / "results" / "tables"
FIGURES = ROOT / "results" / "figures"
PROCESSED = ROOT / "data" / "processed"

pd.set_option("display.max_colwidth", 60)

## 1. 가족 SNP 기초 통계

구성원별 총 SNP 수, 결측률, 이형접합률. Child 2/3의 이형접합률이 다른 구성원보다 뚜렷이
낮은데, 이는 유전체칩 버전 차이로 추정되는 QC 관찰이다(생물학적 차이가 아님).

In [ ]:
family_snp_summary = pd.read_csv(TABLES / "family_snp_summary.csv")
family_snp_summary

## 2. ClinVar 매칭 결과

가족 rsid 합집합(1,117,586개)을 NCBI ClinVar와 매칭한 결과다. 대다수는 Benign(양성)이며,
병원성(Pathogenic) 계열은 소수라는 점을 먼저 확인한다 — "매칭됐다"가 "병원성이다"를
뜻하지 않는다.

In [ ]:
clinvar_summary = pd.read_csv(TABLES / "family_clinvar_summary.csv")
top_significances = (
    clinvar_summary.groupby("clinical_significance")["variant_count"]
    .sum()
    .sort_values(ascending=False)
    .head(8)
)
top_significances

## 3. KEGG 관계 그래프

KEGG 5개 flat file을 구조화해서 만든 69,546건의 엔트리 간 관계를 타입별로 집계한 것이다.

In [ ]:
kegg_type_summary = pd.read_csv(TABLES / "kegg_graph_type_summary.csv")
kegg_type_summary.head(10)

## 4. CFH 사례 연구

가족이 실제로 보유한 ClinVar 병원성(비상충) 변이 4건 중, KEGG `variant.txt`의 명시적
질병 링크와도 교차검증을 통과한 유전자는 **CFH**뿐이었다. 전체 서술은
[`results/case_studies/CFH_case_study.md`](../results/case_studies/CFH_case_study.md).

In [ ]:
cfh_profile = pd.read_csv(TABLES / "cfh_family_disease_profile.csv")
cfh_profile[["disease_id", "disease_name", "cfh_pathway_genes"]]

## 5. 약물유전체(PGx) 사례

전 가족(5명) 공통으로 확인된 대표 PGx 마커는 VKORC1(warfarin), CYP3A5(tacrolimus),
UGT1A1(irinotecan)이다 — 전부 실제 CPIC/FDA 가이드라인에 등재된 유전자-약물 쌍이다.
전체 서술은
[`results/case_studies/pharmacogenomics_case_study.md`](../results/case_studies/pharmacogenomics_case_study.md).

In [ ]:
pgx_summary = pd.read_csv(TABLES / "family_pharmacogenomics_summary.csv")
pgx_summary

## 6. 그림

`results/figures/`에 저장된 정적 그림 4종 중 CFH 가족 유전형과 PGx 히트맵을 확인한다.

In [ ]:
display(Image(filename=str(FIGURES / "cfh_family_genotype.png")))
display(Image(filename=str(FIGURES / "pgx_family_heatmap.png")))

## 7. 한계

- 가족 데이터는 WES/WGS가 아니라 SNP 어레이(23andMe) 기반이라 커버리지가 제한적이다
- 표본 크기가 n=5라 인구 집단 수준의 결론을 낼 수 없다
- 표현형(병력) 정보가 없어 유전형-표현형 상관관계를 임상적으로 검증할 수 없다
- ClinVar "Pathogenic" 표시가 곧 발병을 의미하지 않는다(접합성·침투도·리뷰 신뢰도 등 고려 필요)
- PGx 결과는 마커 식별 수준이며, star-allele/대사자 표현형/실제 용량 계산은 하지 않았다

자세한 내용은 [`docs/limitations.md`](../docs/limitations.md).

## 8. 핵심 요약

- KEGG 5개 flat file → 69,546건의 구조화된 관계로 변환
- 가족 rsid 1,117,586개 중 ClinVar와 59,501건 매칭
- 가족이 실제 보유한 병원성 변이 중 **CFH**만 KEGG 자체 질병 데이터베이스로도 교차검증됨
- 전 가족 공통 PGx 마커(VKORC1/CYP3A5/UGT1A1)가 실제 CPIC/FDA 등재 마커와 일치
- 모든 결과는 가설 생성용 스크리닝이며, 임상 진단·처방의 근거가 아니다